In [ ]:
import requests
from urllib.parse import unquote
import re
import pygsheets

In [ ]:
client = pygsheets.authorize(service_account_file=r'C:\Users\fumio\Documents\gcp-project-365616-6b0f83cca4c2.json')

ss = client.open_by_url('https://docs.google.com/spreadsheets/d/1H-otjPW382f7cUNvaC_p5BJCMw7nmlmuhu2JiNBx_Y0/edit?gid=195125089#gid=195125089')

ws = ss.worksheet_by_title('Nomes Para SeekLoc')

In [ ]:
df = ws.get_as_df()

In [ ]:
len(df)

In [ ]:
df_filtrado = df[df['Telefone'] == '']
df_filtrado = df_filtrado[df_filtrado['Proprietário'] != '']
df_filtrado = df_filtrado[df_filtrado['Whatsapp'] == '']
df_filtrado = df_filtrado[df_filtrado['Anotações Automação'] == '']
df_filtrado = df_filtrado[df_filtrado['Contato Feito'] == 'FALSE']

In [ ]:
len(df_filtrado)

In [ ]:
headers = {
    "accept": "text/javascript, text/html, application/xml, text/xml, */*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "x-prototype-version": "1.6.0",
    "x-requested-with": "XMLHttpRequest",
    "user-agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "referrer": "http://200.201.193.100/seekloc/sistema.php",
    "cookie" : "supremeseekloc=c9jt2o3opbbfn9crcj1ve0re64",
    "referrerPolicy": "strict-origin-when-cross-origin",
}

In [ ]:
def verificar_existencia(nome_cpf) -> str:
    if ' - ' not in nome_cpf:
        return 'Sem início de CPF'

    nome, inicio_cpf = nome_cpf.split(' - ')

    params = {
        "action" : "getselect",
        "tipo" : "form_nome",
        "nome" : nome,
        "mae" : "",
        "uf" : "",
        "cidade" : "",
    }

    try:
        response = requests.get(
            url='http://200.201.193.100/seekloc/ajax.php',
            params = params,
            headers = headers,
        )

    except Exception as e:
        print(f'Erro Request - {e}')
        return f'Erro Request - {e}'

    response_text = unquote(response.content.decode())

    documentos = re.findall(r'DOC:\+(\d*)', response_text)
    get_dados = re.findall(r'getdados\((\d+)\)', response_text)

    if len(documentos) != len(get_dados):
        return 'len(documentos) != len(get_dados)'
    
    for documento, get_dado in zip(documentos, get_dados):
        if f'0{inicio_cpf}' in documento:
            return f'{documento} - {get_dado}'
        
    return 'Erro'


In [ ]:
df_filtrado['Anotações Automação'] = df_filtrado['Proprietário'].apply(verificar_existencia)

In [ ]:
df_filtrado

In [ ]:
df.update(df_filtrado[['Anotações Automação']])

In [ ]:
df

In [ ]:
ws.set_dataframe(
    df,
    'A1'
)